# Catchment and stream network delineation for each dam

Delineates one catchment and its stream network per dam outlet from the conditioned NASADEM DEM,
using pysheds D8 flow routing. All parameters are read from `config/dams.yaml`.
Results are written to `outputs/drainage.gpkg`.

## Load config and DEM

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import shapes
from rasterio.transform import array_bounds
import yaml
from pyproj import CRS, Transformer
from pysheds.grid import Grid
from pysheds.sview import Raster
from shapely.geometry import Point, shape
from shapely.ops import unary_union

ROOT = Path.cwd().parent
CONFIG_PATH = ROOT / "config" / "dams.yaml"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

dem_path = ROOT / "data" / f"dem_{config['dem']['source'].lower()}_utm.tif"

In [ ]:
# Read the nodata value stored in the DEM file
with rasterio.open(dem_path) as src:
    dem_nodata = src.nodata
    dem_crs = CRS.from_user_input(src.crs)

# Load the DEM with pysheds, passing the nodata value so those cells are excluded from routing
grid = Grid.from_raster(str(dem_path), nodata=dem_nodata)
dem = grid.read_raster(str(dem_path), nodata=dem_nodata)

project_crs = CRS.from_user_input(config["crs"])
print(f"DEM: {dem_path.name}, shape {dem.shape}, nodata {dem_nodata}")
print(f"DEM CRS matches project CRS: {dem_crs == project_crs}")

## Condition the DEM

In [ ]:
# Fill single-cell pits, then larger depressions, then resolve flat areas so every cell drains
pit_filled_dem = grid.fill_pits(dem)
flooded_dem = grid.fill_depressions(pit_filled_dem)
inflated_dem = grid.resolve_flats(flooded_dem)

## Flow direction (D8) and flow accumulation

In [ ]:
fdir = grid.flowdir(inflated_dem, routing="d8")
acc = grid.accumulation(fdir, routing="d8")

## Stream threshold

In [ ]:
# Convert the stream and snap thresholds from km² to a number of cells using the DEM pixel area
pixel_area_m2 = abs(grid.affine.a * grid.affine.e)


def km2_to_cells(area_km2):
    """Return the number of DEM cells covering the given area."""
    return int(round(area_km2 * 1e6 / pixel_area_m2))


threshold_km2 = config["stream_threshold_km2"]
threshold_cells = km2_to_cells(threshold_km2)
snap_threshold_km2 = config["snap_threshold_km2"]
snap_threshold_cells = km2_to_cells(snap_threshold_km2)
print(f"Pixel area: {pixel_area_m2:.1f} m²")
print(f"Default stream threshold: {threshold_km2} km² = {threshold_cells} cells")
print(f"Snap threshold: {snap_threshold_km2} km² = {snap_threshold_cells} cells")

# Stream threshold for each dam: its own value if set, otherwise the default
dam_threshold_km2 = {
    d["name"]: d.get("stream_threshold_km2", threshold_km2) for d in config["dams"]
}
for name, value in dam_threshold_km2.items():
    print(f"  {name}: {value} km² = {km2_to_cells(value)} cells")

# Grid settings for boolean masks, with nodata set to False instead of the DEM nodata value
mask_view = grid.viewfinder.copy()
mask_view.nodata = False

stream_mask = Raster(np.asarray(acc >= threshold_cells), viewfinder=mask_view)
snap_mask = Raster(np.asarray(acc >= snap_threshold_cells), viewfinder=mask_view)

## Outlets

In [ ]:
names = [d["name"] for d in config["dams"]]
xy = np.array([[d["easting"], d["northing"]] for d in config["dams"]], dtype=float)

# Reproject the dam coordinates to the project CRS if they are given in a different CRS
coords_crs = CRS.from_user_input(config["coords_crs"])
if coords_crs != project_crs:
    transformer = Transformer.from_crs(coords_crs, project_crs, always_xy=True)
    xy[:, 0], xy[:, 1] = transformer.transform(xy[:, 0], xy[:, 1])

# Snap each outlet to the nearest cell at or above the snap threshold.
# snap_to_mask returns the top-left corner of that cell, so shift it to the cell centre.
corner_xy = grid.snap_to_mask(snap_mask, xy)
snapped_xy = corner_xy + np.array([grid.affine.a / 2, grid.affine.e / 2])
snap_dist_m = np.hypot(*(snapped_xy - xy).T)
flagged = snap_dist_m > config["max_snap_distance_m"]

# Row and column of each snapped outlet cell
outlet_cells = [grid.nearest_cell(x, y, snap="center") for x, y in snapped_xy]

for name, dist, flag in zip(names, snap_dist_m, flagged):
    print(f"{name}: snapped {dist:.1f} m" + ("  FLAGGED" if flag else ""))

## Catchments

In [ ]:
# Cells that are nodata in the DEM
nodata_cells = np.asarray(dem == dem_nodata)


def touches_edge_or_nodata(mask):
    """Return True if the mask reaches the DEM border or is next to a nodata cell."""
    if mask[0, :].any() or mask[-1, :].any() or mask[:, 0].any() or mask[:, -1].any():
        return True
    # Grow the mask by one cell in all 8 directions and test for overlap with nodata
    padded = np.pad(mask, 1)
    grown = np.zeros_like(mask)
    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            grown |= padded[1 + dr : 1 + dr + mask.shape[0], 1 + dc : 1 + dc + mask.shape[1]]
    return bool((grown & nodata_cells).any())


# Length of the D8 step out of each cell: the cell size for N, E, S and W, and the cell size
# times sqrt(2) for the diagonal directions (dirmap order N, NE, E, SE, S, SW, W, NW)
cell_size = abs(grid.affine.a)
dirmap = (64, 128, 1, 2, 4, 8, 16, 32)
diagonal_dirs = [dirmap[k] for k in (1, 3, 5, 7)]
step_length_m = np.where(np.isin(np.asarray(fdir), diagonal_dirs), cell_size * np.sqrt(2), cell_size)
step_weights = Raster(step_length_m.astype(np.float64), viewfinder=fdir.viewfinder)

# Cell slope in percent from the conditioned DEM, with DEM nodata cells set to NaN
conditioned = np.where(nodata_cells, np.nan, np.asarray(inflated_dem, dtype=float))
dz_dy, dz_dx = np.gradient(conditioned, cell_size, cell_size)
slope_pct = np.hypot(dz_dx, dz_dy) * 100

dem_values = np.asarray(dem, dtype=float)

catchment_masks = {}
catchment_records = []
for name, (x, y) in zip(names, snapped_xy):
    catch = grid.catchment(x=x, y=y, fdir=fdir, routing="d8", xytype="coordinate", snap="center")
    mask = np.asarray(catch).astype(bool)
    catchment_masks[name] = mask

    # Longest D8 flow path from any catchment cell to the outlet, in metres
    flow_distance = grid.distance_to_outlet(x=x, y=y, fdir=fdir, weights=step_weights,
                                            routing="d8", xytype="coordinate", snap="center")
    flow_distance = np.asarray(flow_distance, dtype=float)
    longest_path_m = np.nanmax(np.where(mask & np.isfinite(flow_distance), flow_distance, np.nan))

    # Outlets of the other dams that fall inside this catchment
    contained = [
        other for other, (col, row) in zip(names, outlet_cells)
        if other != name and mask[row, col]
    ]

    # Merge the polygons of all catchment cells into one geometry
    polygons = [
        shape(geom)
        for geom, value in shapes(mask.astype(np.uint8), mask=mask, transform=grid.affine)
        if value == 1
    ]
    catchment_records.append({
        "name": name,
        "area_km2": mask.sum() * pixel_area_m2 / 1e6,
        "touches_edge": touches_edge_or_nodata(mask),
        "contains_outlets": ", ".join(contained),
        "stream_threshold_km2": dam_threshold_km2[name],
        "longest_flow_path_km": longest_path_m / 1000,
        "mean_slope_pct": np.nanmean(slope_pct[mask]),
        "elev_min_m": dem_values[mask].min(),
        "elev_max_m": dem_values[mask].max(),
        "geometry": unary_union(polygons),
    })

catchments = gpd.GeoDataFrame(catchment_records, geometry="geometry", crs=project_crs)
catchments.drop(columns="geometry")

## Streams

In [ ]:
# Extract the stream network inside each catchment from cells at or above that dam's stream threshold
stream_frames = []
for name, mask in catchment_masks.items():
    dam_stream_cells = np.asarray(acc) >= km2_to_cells(dam_threshold_km2[name])
    catchment_streams = Raster(dam_stream_cells & mask, viewfinder=mask_view)
    network = grid.extract_river_network(fdir, catchment_streams, routing="d8")
    segments = gpd.GeoDataFrame.from_features(network["features"], crs=project_crs)
    segments = segments[["geometry"]].assign(name=name)

    # Clip the segments to the catchment polygon so no stream extends past the outlet
    catchment_polygon = catchments.loc[catchments["name"] == name, "geometry"].iloc[0]
    segments = gpd.clip(segments, catchment_polygon, keep_geom_type=True)
    stream_frames.append(segments)

streams = gpd.GeoDataFrame(
    pd.concat(stream_frames, ignore_index=True), geometry="geometry", crs=project_crs
)[["name", "geometry"]]
streams.groupby("name").size().rename("segments")

## Save to GeoPackage

In [ ]:
outlets = gpd.GeoDataFrame(
    {
        "name": names,
        "orig_easting": xy[:, 0],
        "orig_northing": xy[:, 1],
        "snap_easting": snapped_xy[:, 0],
        "snap_northing": snapped_xy[:, 1],
        "snap_dist_m": snap_dist_m,
        "flagged": flagged,
    },
    geometry=[Point(x, y) for x, y in snapped_xy],
    crs=project_crs,
)

gpkg_path = OUTPUT_DIR / "drainage.gpkg"
gpkg_path.unlink(missing_ok=True)
outlets.to_file(gpkg_path, layer="outlets", driver="GPKG")
catchments.to_file(gpkg_path, layer="catchments", driver="GPKG")
streams.to_file(gpkg_path, layer="streams", driver="GPKG")
print(f"Saved {gpkg_path}")
outlets.drop(columns="geometry")

## Quick-look plot

In [ ]:
# Show the DEM, catchment outlines, streams and snapped outlets, zoomed to the catchments
left, bottom, right, top = array_bounds(*dem.shape, grid.affine)
dem_display = np.where(nodata_cells, np.nan, np.asarray(dem))

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(dem_display, extent=(left, right, bottom, top), cmap="terrain")
catchments.boundary.plot(ax=ax, color="black", linewidth=1)
streams.plot(ax=ax, color="tab:blue", linewidth=0.6)
outlets.plot(ax=ax, color="red", markersize=30, zorder=3)
for name, (x, y) in zip(names, snapped_xy):
    ax.annotate(name, (x, y), xytext=(5, 5), textcoords="offset points")

xmin, ymin, xmax, ymax = catchments.total_bounds
margin = 0.05 * max(xmax - xmin, ymax - ymin)
ax.set_xlim(xmin - margin, xmax + margin)
ax.set_ylim(ymin - margin, ymax + margin)
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.set_title("Dam catchments and stream networks")
plt.show()

## Diagnostic: flagged outlets

Zoomed view around each outlet whose snap distance is above `max_snap_distance_m`, with log flow accumulation as the background, the stream cells at or above the threshold, the original dam point and the snapped point.

In [ ]:
# Half the width of the square window drawn around each outlet, in metres
half_window_m = 750

log_acc = np.log10(np.asarray(acc, dtype=float) + 1)
stream_cells = np.asarray(stream_mask)
inverse_affine = ~grid.affine

for i in np.flatnonzero(flagged):
    x0, y0 = xy[i]

    # Row and column limits of the window, clipped to the DEM
    col_min, row_min = inverse_affine * (x0 - half_window_m, y0 + half_window_m)
    col_max, row_max = inverse_affine * (x0 + half_window_m, y0 - half_window_m)
    col_min, row_min = max(int(np.floor(col_min)), 0), max(int(np.floor(row_min)), 0)
    col_max = min(int(np.ceil(col_max)), log_acc.shape[1])
    row_max = min(int(np.ceil(row_max)), log_acc.shape[0])

    window = (slice(row_min, row_max), slice(col_min, col_max))
    west, north = grid.affine * (col_min, row_min)
    east, south = grid.affine * (col_max, row_max)
    extent = (west, east, south, north)

    fig, ax = plt.subplots(figsize=(8, 8))
    image = ax.imshow(log_acc[window], extent=extent, cmap="Greys")
    fig.colorbar(image, ax=ax, shrink=0.7, label="log10(flow accumulation + 1), cells")

    # Stream cells at or above the threshold, drawn in blue over the accumulation
    streams_overlay = np.ma.masked_where(~stream_cells[window], stream_cells[window])
    ax.imshow(streams_overlay, extent=extent, cmap="winter", alpha=0.6, interpolation="nearest")

    ax.scatter(*xy[i], color="red", s=60, zorder=3, label="original dam point")
    ax.scatter(*snapped_xy[i], color="yellow", edgecolor="black", s=60, zorder=3,
               label="snapped point")
    ax.plot(*zip(xy[i], snapped_xy[i]), color="red", linestyle="--", linewidth=1)

    ax.set_xlim(west, east)
    ax.set_ylim(south, north)
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    ax.set_title(f"{names[i]}: snapped {snap_dist_m[i]:.0f} m")
    ax.legend(loc="lower left")
    plt.show()

## Diagnostic: highest accumulation near the original point

For Nairutia and Kabunda, finds the cell with the highest flow accumulation within several radii of the original dam coordinates, and the catchment area an outlet at that cell would have.

In [ ]:
from IPython.display import display

diagnostic_dams = ["Nairutia", "Kabunda"]
search_radii_m = [50, 100, 150, 200, 300]

acc_values = np.asarray(acc, dtype=float)
inverse_affine = ~grid.affine


def cells_within(x0, y0, radius_m):
    """Return the rows, columns and cell-centre coordinates of cells whose centre is within radius_m."""
    col_min, row_min = inverse_affine * (x0 - radius_m, y0 + radius_m)
    col_max, row_max = inverse_affine * (x0 + radius_m, y0 - radius_m)
    rows, cols = np.mgrid[
        max(int(np.floor(row_min)), 0) : min(int(np.ceil(row_max)) + 1, acc_values.shape[0]),
        max(int(np.floor(col_min)), 0) : min(int(np.ceil(col_max)) + 1, acc_values.shape[1]),
    ]
    cx, cy = grid.affine * (cols + 0.5, rows + 0.5)
    inside = np.hypot(cx - x0, cy - y0) <= radius_m
    return rows[inside], cols[inside], cx[inside], cy[inside]


best_cells = {}
for name in diagnostic_dams:
    i = names.index(name)
    x0, y0 = xy[i]
    records = []
    for radius in search_radii_m:
        rows, cols, cx, cy = cells_within(x0, y0, radius)
        k = np.argmax(acc_values[rows, cols])
        bx, by = cx[k], cy[k]

        # Catchment area if the outlet were placed at this cell
        catch = grid.catchment(x=bx, y=by, fdir=fdir, routing="d8", xytype="coordinate", snap="center")
        records.append({
            "radius_m": radius,
            "easting": bx,
            "northing": by,
            "distance_m": np.hypot(bx - x0, by - y0),
            "accumulation_km2": acc_values[rows[k], cols[k]] * pixel_area_m2 / 1e6,
            "catchment_km2": np.asarray(catch).astype(bool).sum() * pixel_area_m2 / 1e6,
        })
    best_cells[name] = pd.DataFrame(records)
    print(f"{name}: original point ({x0:.1f}, {y0:.1f}), current snap {snap_dist_m[i]:.1f} m")
    display(best_cells[name].round({"easting": 1, "northing": 1, "distance_m": 1,
                                     "accumulation_km2": 3, "catchment_km2": 3}))

In [ ]:
from matplotlib.patches import Circle

# Half the width of the square window drawn around each original point, in metres
half_window_m = 500
circle_radii_m = [100, 200, 300]

for name in diagnostic_dams:
    i = names.index(name)
    x0, y0 = xy[i]

    # Row and column limits of the window, clipped to the DEM
    col_min, row_min = inverse_affine * (x0 - half_window_m, y0 + half_window_m)
    col_max, row_max = inverse_affine * (x0 + half_window_m, y0 - half_window_m)
    col_min, row_min = max(int(np.floor(col_min)), 0), max(int(np.floor(row_min)), 0)
    col_max = min(int(np.ceil(col_max)), acc_values.shape[1])
    row_max = min(int(np.ceil(row_max)), acc_values.shape[0])
    west, north = grid.affine * (col_min, row_min)
    east, south = grid.affine * (col_max, row_max)

    fig, ax = plt.subplots(figsize=(8, 8))
    image = ax.imshow(np.log10(acc_values[row_min:row_max, col_min:col_max] + 1),
                      extent=(west, east, south, north), cmap="Greys")
    fig.colorbar(image, ax=ax, shrink=0.7, label="log10(flow accumulation + 1), cells")

    for radius in circle_radii_m:
        ax.add_patch(Circle((x0, y0), radius, fill=False, edgecolor="tab:orange",
                            linestyle="--", linewidth=1))
        ax.annotate(f"{radius} m", (x0, y0 + radius), color="tab:orange", ha="center",
                    va="bottom", fontsize=8)

    # Best cell at each search radius, labelled with the radius
    best = best_cells[name]
    ax.scatter(best["easting"], best["northing"], color="tab:cyan", edgecolor="black", s=50,
               zorder=3, label="highest accumulation within radius")
    for _, row in best.iterrows():
        ax.annotate(f"{row['radius_m']:.0f}", (row["easting"], row["northing"]),
                    xytext=(5, -10), textcoords="offset points", fontsize=8, color="tab:blue")

    ax.scatter(x0, y0, color="red", s=60, zorder=4, label="original dam point")
    ax.scatter(*snapped_xy[i], color="yellow", edgecolor="black", s=60, zorder=4,
               label="current snapped point")

    ax.set_xlim(west, east)
    ax.set_ylim(south, north)
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    ax.set_title(f"{name}: highest accumulation within search radii")
    ax.legend(loc="lower left", fontsize=8)
    plt.show()